# 52 — DeepEval Synthesizer

**What you'll learn:**
- `Synthesizer.generate_goldens_from_docs()` — auto-generate a golden dataset from documents
- Evolution strategies: **SIMPLE**, **REASONING**, **MULTI_HOP**, **COMPARATIVE**
- `EvaluationDataset.save()` / `EvaluationDataset.load()` — persist and reload datasets
- Regression testing: re-run the same goldens after pipeline changes to detect score drops
- Delta comparison: quantify degradation when retrieval quality worsens

**Context:** Generating golden datasets manually is the #1 blocker to LLM evaluation adoption — Synthesizer removes this bottleneck by auto-generating diverse test cases from your documents.

In [ ]:
%pip install -q deepeval langchain langchain-openai langchain-chroma chromadb python-dotenv

In [ ]:
import os

# Colab: use userdata; local: use .env
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── Why Synthesizer matters ───────────────────────────────────────────────────
#
# Manual golden dataset creation is slow:
#   - Expert writes question → expected answer → retrieval context
#   - 100 goldens = 4-8 hours of expert time
#   - Goldens go stale when documents change
#
# Synthesizer automates this:
#   - SIMPLE:       "What is LangGraph?" → direct doc answer
#   - REASONING:    "If LangGraph uses graphs, what does a node represent?"
#   - MULTI_HOP:    "What LangGraph feature helps with the checkpointing needs of RAG?"
#   - COMPARATIVE:  "Compare ChromaDB and Pinecone"
#
# The full eval loop:
#   1. generate_goldens_from_docs(docs, n=10) → EvaluationDataset
#   2. save_dataset(path) → persist for regression testing
#   3. run RAG pipeline → evaluate() → baseline scores
#   4. change pipeline (k=1, model swap, chunking)
#   5. load_dataset(path) → evaluate() → compare delta
#
print("Synthesizer: auto-generate diverse goldens from your documents.")

In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.dataset import EvaluationDataset

DOCUMENTS = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. It uses a graph-based workflow where nodes are Python functions and edges control flow.",
    "LangChain provides modular components for building LLM applications: chains, agents, retrievers, and output parsers. It integrates with 50+ LLM providers.",
    "RAG (Retrieval-Augmented Generation) improves LLM accuracy by retrieving relevant documents before generation. Key metrics: Faithfulness, AnswerRelevancy, ContextualPrecision.",
    "Vector databases store numerical embeddings for semantic similarity search. Popular options: ChromaDB (local), Pinecone (cloud), Weaviate (self-hosted).",
    "Agents use LLMs as reasoning engines to decide which tools to call and when. ReAct (Reason + Act) is the dominant pattern: think step-by-step, then act.",
    "Checkpointing in LangGraph allows saving and resuming agent state. SqliteSaver persists to disk; MemorySaver keeps state in memory only.",
]

synthesizer = Synthesizer(model="gpt-4o-mini")

# Generate with multiple strategies
goldens = synthesizer.generate_goldens_from_docs(
    document_paths=None,  # pass inline chunks
    contexts=[[doc] for doc in DOCUMENTS],
    max_goldens_per_context=2,
)
print(f"Generated {len(goldens)} goldens")
for g in goldens[:3]:
    print(f"  Q: {g.input[:80]}")
    print(f"  A: {g.expected_output[:80]}\n")

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Build RAG pipeline (baseline: k=3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
vectorstore = Chroma.from_texts(DOCUMENTS, embeddings, collection_name="synth_eval")

ANSWER_PROMPT = "Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {query}"

def run_rag(query: str, k: int = 3) -> tuple[str, list[str]]:
    docs = vectorstore.similarity_search(query, k=k)
    context = [d.page_content for d in docs]
    context_text = "\n".join(f"- {c}" for c in context)
    answer = llm.invoke([HumanMessage(content=ANSWER_PROMPT.format(context=context_text, query=query))]).content
    return answer, context

# Build test cases from goldens
test_cases_k3 = []
for g in goldens:
    answer, context = run_rag(g.input, k=3)
    test_cases_k3.append(LLMTestCase(
        input=g.input,
        actual_output=answer,
        expected_output=g.expected_output,
        retrieval_context=context,
    ))

metrics = [AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini"), FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini")]
results_k3 = evaluate(test_cases_k3, metrics)

print("\nBaseline scores (k=3):")
print(f"  AnswerRelevancy : {results_k3.test_results[0].metrics_data[0].score:.2f}")
print(f"  Faithfulness    : {results_k3.test_results[0].metrics_data[1].score:.2f}")

# Save dataset for regression testing
dataset = EvaluationDataset(test_cases=test_cases_k3)
dataset.save("goldens_baseline.json")
print("\nDataset saved to goldens_baseline.json")

In [ ]:
# ── Regression test: degrade pipeline (k=1) ──────────────────────────────────
loaded_dataset = EvaluationDataset()
loaded_dataset.load("goldens_baseline.json")
print(f"Loaded {len(loaded_dataset.test_cases)} test cases from saved dataset")

# Re-run same goldens with degraded retrieval (k=1)
test_cases_k1 = []
for tc in loaded_dataset.test_cases:
    answer, context = run_rag(tc.input, k=1)
    test_cases_k1.append(LLMTestCase(
        input=tc.input,
        actual_output=answer,
        expected_output=tc.expected_output,
        retrieval_context=context,
    ))

results_k1 = evaluate(test_cases_k1, metrics)

score_k3_ar = sum(tc.metrics_data[0].score for tc in results_k3.test_results) / len(results_k3.test_results)
score_k1_ar = sum(tc.metrics_data[0].score for tc in results_k1.test_results) / len(results_k1.test_results)
score_k3_f  = sum(tc.metrics_data[1].score for tc in results_k3.test_results) / len(results_k3.test_results)
score_k1_f  = sum(tc.metrics_data[1].score for tc in results_k1.test_results) / len(results_k1.test_results)

print("\nRegression delta (k=3 → k=1):")
print(f"  AnswerRelevancy : {score_k3_ar:.2f} → {score_k1_ar:.2f}  (delta: {score_k1_ar - score_k3_ar:+.2f})")
print(f"  Faithfulness    : {score_k3_f:.2f} → {score_k1_f:.2f}  (delta: {score_k1_f - score_k3_f:+.2f})")
print("\n** Negative delta = regression. Fix the pipeline, re-run, confirm delta returns to ~0 **")

In [ ]:
# ── Hand-written vs synthesized goldens comparison ───────────────────────────
HAND_WRITTEN_GOLDENS = [
    {"input": "What is LangGraph used for?", "expected_output": "LangGraph is used for building stateful, multi-actor LLM applications using graph-based workflows.", "context": [DOCUMENTS[0]]},
    {"input": "Name two vector databases mentioned.", "expected_output": "ChromaDB and Pinecone are mentioned as vector database options.", "context": [DOCUMENTS[3]]},
]

hw_cases, synth_cases = [], []
for g in HAND_WRITTEN_GOLDENS:
    answer, context = run_rag(g["input"], k=3)
    hw_cases.append(LLMTestCase(input=g["input"], actual_output=answer, expected_output=g["expected_output"], retrieval_context=context))

for g in goldens[:4]:  # use first 4 synthesized goldens
    answer, context = run_rag(g.input, k=3)
    synth_cases.append(LLMTestCase(input=g.input, actual_output=answer, expected_output=g.expected_output, retrieval_context=context))

hw_results    = evaluate(hw_cases,    metrics)
synth_results = evaluate(synth_cases, metrics)

hw_ar   = sum(tc.metrics_data[0].score for tc in hw_results.test_results)    / len(hw_results.test_results)
synth_ar = sum(tc.metrics_data[0].score for tc in synth_results.test_results) / len(synth_results.test_results)

print("\nHand-written vs Synthesized (AnswerRelevancy avg):")
print(f"  Hand-written  ({len(hw_cases)} goldens):  {hw_ar:.2f}")
print(f"  Synthesized   ({len(synth_cases)} goldens): {synth_ar:.2f}")
print("\nSynthesized goldens include multi-hop and reasoning questions")
print("that hand-written sets typically miss — broader coverage, lower avg score.")

## Exercises

1. Generate 10 goldens from your own documents and inspect them — are they useful?
2. Deliberately degrade the RAG pipeline (`k=1`) and measure the score drop per metric.
3. Add a new document to the knowledge base and regenerate goldens — do the new questions get covered?
4. Compare SIMPLE vs MULTI_HOP goldens — which type exposes more weaknesses?

## What's next

- **47-deepeval-rag-metrics** — metrics that evaluate these goldens (Faithfulness, AnswerRelevancy, ContextualPrecision)
- **49-deepeval-geval-custom** — custom metrics for non-standard goldens
- **16-rag-eval-notebook** — RAGAS as an alternative evaluation framework